In [1]:
import pandas as pd
import plotly.express as px

safety_info_stats = pd.read_csv(
    "../data/safety_stats.csv"
)

In [2]:
display(safety_info_stats.head())
print(safety_info_stats.shape)
print(safety_info_stats.columns)

print("컬럼:", safety_info_stats.columns.tolist())
print("연도 목록:")
print(sorted(safety_info_stats["year"].unique()))

print("연도 개수:", safety_info_stats["year"].nunique())

,iso3,country_kr,year,count
0,AFG,아프가니스탄,2011,1
1,AFG,아프가니스탄,2012,3
2,AFG,아프가니스탄,2013,3
3,AFG,아프가니스탄,2014,18
4,AFG,아프가니스탄,2015,42


(1134, 4)
Index(['iso3', 'country_kr', 'year', 'count'], dtype='str')
컬럼: ['iso3', 'country_kr', 'year', 'count']
연도 목록:
[np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
연도 개수: 15


In [3]:
selected_year = 2024

top10 = (
    safety_info_stats[safety_info_stats["year"] == selected_year]
    .sort_values("count", ascending=False)
    .head(10)
)

display(top10)

,iso3,country_kr,year,count
157,CHL,칠레,2024,3
169,CHN,중국,2024,2
256,ECU,에콰도르,2024,2
218,DEU,독일,2024,2
781,PNG,파푸아뉴기니,2024,2
44,AUS,호주,2024,1
83,BGD,방글라데시,2024,1
116,BOL,볼리비아,2024,1
195,CUB,쿠바,2024,1
33,ARG,아르헨티나,2024,1


In [4]:
fig = px.bar(
    top10,
    x="count",
    y="country_kr",
    orientation="h",
    title=f"{selected_year}년 국가별 안전정보 건수 TOP 10",
    labels={
        "count": "안전정보 건수",
        "country_kr": "국가"
    },
    text="count"
)

# 안전정보가 많은 국가가 위쪽에 오도록 정렬
fig.update_layout(
    yaxis={
        "categoryorder": "total ascending"
    },
    xaxis={
        "dtick": 1
    }
)

fig.show()

In [5]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

years = sorted(safety_info_stats["year"].unique())
years = [year for year in years if year >= 2011]

# 각 연도별 제목 만들기
subplot_titles = []

for year in years:
    year_data = safety_info_stats[
        safety_info_stats["year"] == year
    ]

    # 해당 연도 전체 안전정보 건수
    total_count = year_data["count"].sum()

    # 해당 연도 TOP 10 안전정보 건수
    top10_count = (
        year_data
        .nlargest(10, "count")["count"]
        .sum()
    )

    subplot_titles.append(
        f"{year}년 | 전체 {total_count}건 · TOP10 {top10_count}건"
    )


fig = make_subplots(
    rows=5,
    cols=3,
    subplot_titles=subplot_titles,
    vertical_spacing=0.06,
    horizontal_spacing=0.08
)


for i, year in enumerate(years):
    row = i // 3 + 1
    col = i % 3 + 1

    top10 = (
        safety_info_stats[safety_info_stats["year"] == year]
        .sort_values("count", ascending=False)
        .head(10)
        .sort_values("count", ascending=True)
    )

    fig.add_trace(
        go.Bar(
            x=top10["count"],
            y=top10["country_kr"],
            orientation="h",
            text=top10["count"],
            textposition="outside",
            showlegend=False
        ),
        row=row,
        col=col
    )

fig.update_layout(
    height=1800,
    width=1200,
    title="2011~2024년 연도별 국가 안전정보 등록 건수 TOP 10",
    showlegend=False
)

fig.show()


# 폴더 생성 후 이미지 저장

from pathlib import Path

Path("../outputs/charts").mkdir(parents=True, exist_ok=True)

fig.write_image(
    "../outputs/charts/safety_info_top10_by_year.png",
    width=1200,
    height=1800,
    scale=2
)

분석 참고사항

연도별 안전정보 등록 건수의 편차가 큰 것을 확인함.
2021년 이후 등록 건수가 크게 감소하지만, 데이터만으로 감소 원인을 특정하기 어려움.
따라서 연도별 건수 차이를 실제 국가 위험도의 증감으로 해석하지 않음.
2024년 데이터는 7월 23일까지 수록되어 있어 연간 데이터가 아님.
최종 시각화는 연도 간 비교보다 선택한 연도 내 국가별 TOP10 비교를 목적으로 사용함.